# Fine-tune PhoBERT cho Vietnamese Banking Assistant

**Mục tiêu:** Fine-tune `vinai/phobert-base` trên dataset `undertheseanlp/UTS2017_Bank`
(phản hồi khách hàng ngân hàng, tiếng Việt gốc) để tạo 2 model nhỏ dùng trong
pipeline agent sau này:

1. **Aspect Classifier** (14 lớp) — phân loại feedback thuộc khía cạnh nào
   (CUSTOMER_SUPPORT, CARD, LOAN, INTEREST_RATE, ...) → dùng để **routing** trong LangGraph agent.
2. **Sentiment Classifier** (3 lớp: positive/negative/neutral) → dùng để quyết định
   **escalate** (feedback tiêu cực nghiêm trọng) hay xử lý bình thường (RAG/LLM).

Chạy trên **Kaggle Notebook** với GPU (T4 x2, free) — Settings > Accelerator > GPU T4 x2.

Checkpoint cuối chỉ ~500MB, có thể push lên HuggingFace Hub, không cần giữ trên máy cá nhân.


## 0. Cài đặt thư viện

In [6]:
# Cài TẤT CẢ package cần dùng trong toàn bộ notebook (bao gồm cả phần RAG ở mục 9) ngay từ đầu,
# TRƯỚC khi có bất kỳ import nào chạy.
#
# QUAN TRỌNG: KHÔNG dùng --upgrade-strategy eager nữa. Mọi package cần nâng cấp (pydantic,
# pydantic-core, opentelemetry-*, langchain*, chromadb...) đều đã được liệt kê tên tường minh
# bên dưới với --upgrade, nên pip vẫn tự nâng đúng các package đó khi cần — không cần eager.
# eager từng gây lỗi torch/torchvision lệch version, và giờ lại gây lỗi tương tự với numpy
# ("cannot import name '_center' from 'numpy._core.umath'") vì nó ép nâng cấp cả các package
# KHÔNG được liệt kê tên (transitive deps) mà môi trường Kaggle vốn đã có sẵn bản tương thích.
#
# Để chắc chắn tuyệt đối, vẫn khóa cứng version các package "nhạy cảm về ABI biên dịch sẵn"
# (torch, torchvision, torchaudio, numpy, scipy) bằng constraints file — dù có eager hay không.
import importlib.metadata as metadata

constraints = []
for pkg in ["torch", "torchvision", "torchaudio", "numpy", "scipy"]:
    try:
        constraints.append(f"{pkg}=={metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        pass

with open("/tmp/constraints.txt", "w") as f:
    f.write("\n".join(constraints))

print("Giữ nguyên version các package sau (không cho pip đổi):")
print(open("/tmp/constraints.txt").read())

!pip install -q --upgrade -c /tmp/constraints.txt transformers datasets evaluate underthesea accelerate huggingface_hub scikit-learn langchain langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-common opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http opentelemetry-semantic-conventions pydantic pydantic-core


Giữ nguyên version các package sau (không cho pip đổi):
torch==2.10.0+cu128
torchvision==0.25.0+cu128
torchaudio==2.10.0+cu128
numpy==2.0.2
scipy==1.16.3


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score
from underthesea import word_tokenize
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## 1. Load dataset `undertheseanlp/UTS2017_Bank`

Dataset có 3 subset. Ta dùng:
- `classification` (14 lớp aspect) — task chính, dùng cho routing.
- `sentiment` (3 lớp) — task phụ, dùng cho escalation logic.


In [8]:
ds_aspect = load_dataset("undertheseanlp/UTS2017_Bank", "classification")
ds_sentiment = load_dataset("undertheseanlp/UTS2017_Bank", "sentiment")

print(ds_aspect)
print(ds_sentiment)
print()
print("Ví dụ aspect:", ds_aspect["train"][0])
print("Ví dụ sentiment:", ds_sentiment["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1977
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 494
    })
})
DatasetDict({
    train: Dataset({
        features: ['text', 'sentiment'],
        num_rows: 1977
    })
    test: Dataset({
        features: ['text', 'sentiment'],
        num_rows: 494
    })
})

Ví dụ aspect: {'text': 'Cần tư vấn mà add  k rep', 'label': 'CUSTOMER_SUPPORT'}
Ví dụ sentiment: {'text': 'Cần tư vấn mà add  k rep', 'sentiment': 'negative'}


In [9]:
# EDA nhanh: phân bố nhãn — dataset khá mất cân bằng (CUSTOMER_SUPPORT ~39%, TRADEMARK ~35%,
# các lớp còn lại đều dưới 5%), cần xử lý ở bước fine-tune (class weighting).
train_df = ds_aspect["train"].to_pandas()
print("Phân bố aspect (train):")
print(train_df["label"].value_counts())
print()

sent_df = ds_sentiment["train"].to_pandas()
print("Phân bố sentiment (train):")
print(sent_df["sentiment"].value_counts())


Phân bố aspect (train):
label
CUSTOMER_SUPPORT    774
TRADEMARK           699
LOAN                 74
INTERNET_BANKING     70
OTHER                69
CARD                 66
INTEREST_RATE        60
PROMOTION            53
DISCOUNT             41
MONEY_TRANSFER       34
PAYMENT              15
SAVING               13
ACCOUNT               5
SECURITY              4
Name: count, dtype: int64

Phân bố sentiment (train):
sentiment
positive    1211
negative     743
neutral       23
Name: count, dtype: int64


## 2. Tiền xử lý: PhoBERT cần văn bản đã tách từ (word-segmented)

PhoBERT được pretrain trên tiếng Việt đã tách từ (VD: "ngân_hàng" thay vì "ngân hàng"),
nên trước khi đưa vào tokenizer của PhoBERT, ta phải chạy qua `underthesea.word_tokenize`
để tách từ đúng chuẩn. Bỏ qua bước này là lỗi rất phổ biến khi dùng PhoBERT — làm giảm
độ chính xác đáng kể vì input không khớp phân phối lúc pretrain.


In [10]:
def segment(text: str) -> str:
    tokens = word_tokenize(text)
    return " ".join(tokens)

# demo
print(segment("Hotline khó gọi quá gọi mãi ko thưa máy à"))


Hotline khó gọi quá gọi mãi ko thưa máy à


In [11]:
def preprocess_aspect(example):
    example["text_seg"] = segment(example["text"])
    return example

def preprocess_sentiment(example):
    example["text_seg"] = segment(example["text"])
    return example

ds_aspect = ds_aspect.map(preprocess_aspect)
ds_sentiment = ds_sentiment.map(preprocess_sentiment)


Map:   0%|          | 0/1977 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

Map:   0%|          | 0/1977 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

## 3. Encode nhãn + tokenize bằng PhoBERT tokenizer

In [12]:
MODEL_NAME = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Aspect labels ---
aspect_labels = sorted(set(ds_aspect["train"]["label"]))
aspect2id = {l: i for i, l in enumerate(aspect_labels)}
id2aspect = {i: l for l, i in aspect2id.items()}
print("Aspect classes:", aspect_labels)

def encode_aspect(example):
    enc = tokenizer(example["text_seg"], truncation=True, max_length=256)
    enc["labels"] = aspect2id[example["label"]]
    return enc

ds_aspect_enc = ds_aspect.map(encode_aspect, remove_columns=ds_aspect["train"].column_names)

# --- Sentiment labels ---
sentiment_labels = sorted(set(ds_sentiment["train"]["sentiment"]))
sent2id = {l: i for i, l in enumerate(sentiment_labels)}
id2sent = {i: l for l, i in sent2id.items()}
print("Sentiment classes:", sentiment_labels)

def encode_sentiment(example):
    enc = tokenizer(example["text_seg"], truncation=True, max_length=256)
    enc["labels"] = sent2id[example["sentiment"]]
    return enc

ds_sentiment_enc = ds_sentiment.map(encode_sentiment, remove_columns=ds_sentiment["train"].column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Aspect classes: ['ACCOUNT', 'CARD', 'CUSTOMER_SUPPORT', 'DISCOUNT', 'INTEREST_RATE', 'INTERNET_BANKING', 'LOAN', 'MONEY_TRANSFER', 'OTHER', 'PAYMENT', 'PROMOTION', 'SAVING', 'SECURITY', 'TRADEMARK']


Map:   0%|          | 0/1977 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

Sentiment classes: ['negative', 'neutral', 'positive']


Map:   0%|          | 0/1977 [00:00<?, ? examples/s]

Map:   0%|          | 0/494 [00:00<?, ? examples/s]

## 4. Xử lý mất cân bằng lớp (class imbalance)

Aspect dataset lệch mạnh (CUSTOMER_SUPPORT + TRADEMARK chiếm ~75% dữ liệu), nếu train
bình thường model sẽ có xu hướng đoán nghiêng về 2 lớp lớn. Dùng **weighted CrossEntropyLoss**
qua `class_weight='balanced'` của sklearn, custom vào Trainer.


In [13]:
def get_class_weights(encoded_labels, num_labels):
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=encoded_labels,
    )
    return torch.tensor(weights, dtype=torch.float)

aspect_class_weights = get_class_weights(ds_aspect_enc["train"]["labels"], len(aspect_labels))
sentiment_class_weights = get_class_weights(ds_sentiment_enc["train"]["labels"], len(sentiment_labels))
print("Aspect class weights:", aspect_class_weights)
print("Sentiment class weights:", sentiment_class_weights)


class WeightedTrainer(Trainer):
    """Trainer dùng weighted CrossEntropyLoss thay vì loss mặc định."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


Aspect class weights: tensor([28.2429,  2.1396,  0.1824,  3.4443,  2.3536,  2.0173,  1.9083,  4.1534,
         2.0466,  9.4143,  2.6644, 10.8626, 35.3036,  0.2020])
Sentiment class weights: tensor([ 0.8869, 28.6522,  0.5442])


## 5. Fine-tune PhoBERT — Aspect Classifier (14 lớp)

Đây là model chính dùng để **routing** trong LangGraph agent sau này.


In [14]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

model_aspect = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(aspect_labels)
).to(device)

training_args_aspect = TrainingArguments(
    output_dir="./phobert-banking-aspect",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,  # chỉ giữ 2 checkpoint gần nhất, tránh tràn ổ đĩa Kaggle (/kaggle/working ~20GB)
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer_aspect = WeightedTrainer(
    model=model_aspect,
    args=training_args_aspect,
    train_dataset=ds_aspect_enc["train"],
    eval_dataset=ds_aspect_enc["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=aspect_class_weights,
)

trainer_aspect.train()


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=930, training_loss=1.4816426328433456, metrics={'train_runtime': 608.8445, 'train_samples_per_second': 48.707, 'train_steps_per_second': 1.527, 'total_flos': 2387033639207772.0, 'train_loss': 1.4816426328433456, 'epoch': 15.0})

In [15]:
# Đánh giá chi tiết theo từng lớp — quan trọng hơn accuracy tổng vì dataset mất cân bằng.
# Dùng macro F1 để đánh giá công bằng, không bị lệch bởi 2 lớp lớn (CUSTOMER_SUPPORT, TRADEMARK).
preds_output = trainer_aspect.predict(ds_aspect_enc["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(
    y_true, y_pred,
    labels=list(range(len(aspect_labels))),  # ép đủ 14 lớp dù test set thiếu ACCOUNT/SECURITY
    target_names=[id2aspect[i] for i in range(len(aspect_labels))],
    zero_division=0,
))


                  precision    recall  f1-score   support

         ACCOUNT       0.00      0.00      0.00         0
            CARD       0.50      0.66      0.57        44
CUSTOMER_SUPPORT       0.94      0.61      0.74       338
        DISCOUNT       0.07      0.50      0.12         2
   INTEREST_RATE       0.17      0.50      0.25         6
INTERNET_BANKING       0.32      0.81      0.46        32
            LOAN       0.38      1.00      0.55         3
  MONEY_TRANSFER       0.00      0.00      0.00         2
           OTHER       0.26      0.50      0.34        12
         PAYMENT       0.00      0.00      0.00         2
       PROMOTION       0.20      0.11      0.14         9
          SAVING       0.00      0.00      0.00         3
        SECURITY       0.00      0.00      0.00         0
       TRADEMARK       0.60      0.61      0.60        41

        accuracy                           0.61       494
       macro avg       0.24      0.38      0.27       494
    weighted

## 6. Fine-tune PhoBERT — Sentiment Classifier (3 lớp)

Dùng cho logic escalation: feedback `negative` + aspect nhạy cảm (VD: SECURITY, ACCOUNT)
→ agent nên escalate thay vì để LLM tự trả lời.


In [16]:
model_sentiment = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(sentiment_labels)
).to(device)

training_args_sentiment = TrainingArguments(
    output_dir="./phobert-banking-sentiment",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,  # chỉ giữ 2 checkpoint gần nhất, tránh tràn ổ đĩa Kaggle (/kaggle/working ~20GB)
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer_sentiment = WeightedTrainer(
    model=model_sentiment,
    args=training_args_sentiment,
    train_dataset=ds_sentiment_enc["train"],
    eval_dataset=ds_sentiment_enc["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=sentiment_class_weights,
)

trainer_sentiment.train()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transf

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.813534,1.042176,0.876518,0.583745,0.868407
2,0.586739,0.836322,0.925101,0.619096,0.917127
3,0.286638,1.031753,0.933198,0.625042,0.925282
4,0.324224,1.232901,0.931174,0.623809,0.923624
5,0.180373,1.483474,0.935223,0.627128,0.927682
6,0.191804,1.129449,0.927126,0.682885,0.923486
7,0.300200,1.160046,0.925101,0.621987,0.920391
8,0.221604,1.493456,0.927126,0.622647,0.921499
9,0.195273,1.445855,0.933198,0.627072,0.927581
10,0.044296,1.353498,0.931174,0.626488,0.926513


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=620, training_loss=0.39715168428036474, metrics={'train_runtime': 407.0198, 'train_samples_per_second': 48.573, 'train_steps_per_second': 1.523, 'total_flos': 1582142606163378.0, 'train_loss': 0.39715168428036474, 'epoch': 10.0})

In [17]:
preds_output_s = trainer_sentiment.predict(ds_sentiment_enc["test"])
y_pred_s = np.argmax(preds_output_s.predictions, axis=-1)
y_true_s = preds_output_s.label_ids

print(classification_report(
    y_true_s, y_pred_s,
    labels=list(range(len(sentiment_labels))),  # phòng trường hợp thiếu lớp trong test set
    target_names=[id2sent[i] for i in range(len(sentiment_labels))],
    zero_division=0,
))


              precision    recall  f1-score   support

    negative       0.95      0.93      0.94       301
     neutral       0.33      0.12      0.18         8
    positive       0.90      0.95      0.92       185

    accuracy                           0.93       494
   macro avg       0.73      0.67      0.68       494
weighted avg       0.92      0.93      0.92       494



## 7. Lưu model + đẩy lên HuggingFace Hub

Checkpoint mỗi model chỉ ~500MB (PhoBERT-base). Lưu lên HF Hub để **không cần tải về máy** —
khi cần deploy chỉ cần `from_pretrained("<username>/phobert-banking-aspect")`.

Cần tạo token tại https://huggingface.co/settings/tokens và lưu vào Kaggle Secrets
(Add-ons > Secrets) với tên `HF_TOKEN`, rồi bỏ comment đoạn `push_to_hub` bên dưới.


In [18]:
# Lưu local trước (trong Kaggle, thư mục /kaggle/working được giữ lại khi Save Version)
trainer_aspect.save_model("/kaggle/working/phobert-banking-aspect")
tokenizer.save_pretrained("/kaggle/working/phobert-banking-aspect")

trainer_sentiment.save_model("/kaggle/working/phobert-banking-sentiment")
tokenizer.save_pretrained("/kaggle/working/phobert-banking-sentiment")

# --- Đẩy lên HuggingFace Hub (bỏ comment khi đã có HF_TOKEN trong Kaggle Secrets) ---
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

HF_USERNAME = "minhunhooo"
trainer_aspect.model.push_to_hub(f"{HF_USERNAME}/phobert-banking-aspect")
tokenizer.push_to_hub(f"{HF_USERNAME}/phobert-banking-aspect")

trainer_sentiment.model.push_to_hub(f"{HF_USERNAME}/phobert-banking-sentiment")
tokenizer.push_to_hub(f"{HF_USERNAME}/phobert-banking-sentiment")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/minhunhooo/phobert-banking-sentiment/commit/71e931f327ee2e4641fcd7fe079e812a4dc68d69', commit_message='Upload tokenizer', commit_description='', oid='71e931f327ee2e4641fcd7fe079e812a4dc68d69', pr_url=None, repo_url=RepoUrl('https://huggingface.co/minhunhooo/phobert-banking-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='minhunhooo/phobert-banking-sentiment'), pr_revision=None, pr_num=None)

## 8. Demo inference nhanh — mô phỏng bước routing trong agent

In [19]:
aspect_pipe_model = trainer_aspect.model.eval()
sentiment_pipe_model = trainer_sentiment.model.eval()

@torch.no_grad()
def predict(text: str):
    text_seg = segment(text)
    enc = tokenizer(text_seg, truncation=True, max_length=256, return_tensors="pt").to(device)

    aspect_logits = aspect_pipe_model(**enc).logits
    aspect_pred = id2aspect[int(torch.argmax(aspect_logits, dim=-1))]

    sent_logits = sentiment_pipe_model(**enc).logits
    sent_pred = id2sent[int(torch.argmax(sent_logits, dim=-1))]

    # logic escalation đơn giản — sẽ chuyển thành 1 node trong LangGraph
    escalate = sent_pred == "negative" and aspect_pred in {"SECURITY", "ACCOUNT", "CARD"}

    return {
        "text": text,
        "aspect": aspect_pred,
        "sentiment": sent_pred,
        "escalate": escalate,
    }


examples = [
    "Hotline khó gọi quá gọi mãi ko thưa máy à",
    "Dịch vụ thanh toán tiền điện rất tiện lợi, cảm ơn ngân hàng",
    "Tài khoản của tôi bị trừ tiền lạ mà không rõ lý do, rất lo lắng",
    "Lãi suất tiết kiệm kỳ này cao hơn tháng trước, khá tốt",
]

for ex in examples:
    print(predict(ex))
    print()


{'text': 'Hotline khó gọi quá gọi mãi ko thưa máy à', 'aspect': 'CUSTOMER_SUPPORT', 'sentiment': 'negative', 'escalate': False}

{'text': 'Dịch vụ thanh toán tiền điện rất tiện lợi, cảm ơn ngân hàng', 'aspect': 'PAYMENT', 'sentiment': 'positive', 'escalate': False}

{'text': 'Tài khoản của tôi bị trừ tiền lạ mà không rõ lý do, rất lo lắng', 'aspect': 'MONEY_TRANSFER', 'sentiment': 'negative', 'escalate': False}

{'text': 'Lãi suất tiết kiệm kỳ này cao hơn tháng trước, khá tốt', 'aspect': 'INTEREST_RATE', 'sentiment': 'positive', 'escalate': False}



## 9. Xây dựng RAG Corpus quy mô lớn — Vietnamese Finance/Banking News

Thay vì tự soạn vài chục FAQ, ta lọc ra tập con banking/finance từ
**`vietgpt/binhvq_news_vi`** — 19,4 triệu bài báo tiếng Việt. Dùng `streaming=True`
để **không tải hết 4,78GB về**, chỉ lọc theo từ khóa và giữ lại phần cần dùng.

Kết quả: một RAG corpus quy mô lớn (hàng chục nghìn tài liệu), lấy hoàn toàn từ nguồn
public có sẵn, không cần tự thu thập/scrape.


In [20]:
# Đã cài toàn bộ package RAG (langchain, chromadb, pydantic, opentelemetry...) ngay ở cell cài đặt
# đầu tiên (mục 0), TRƯỚC khi bất kỳ import nào chạy trong notebook — để tránh lỗi cache bộ nhớ
# giữa 2 lần cài. Không cần chạy pip install lại ở đây nữa, cell này để trống có chủ đích.


In [21]:
import itertools
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


In [22]:
# --- 9.1 Lọc bài viết banking/finance từ corpus 19.4M bài, theo streaming ---

KEYWORDS = [
    "ngân hàng", "tài chính", "lãi suất", "tín dụng", "vay vốn", "thẻ tín dụng",
    "tỷ giá", "chứng khoán", "cổ phiếu", "tiết kiệm", "chuyển khoản", "bảo hiểm",
    "vietcombank", "bidv", "techcombank", "vietinbank", "agribank", "vpbank",
    "sacombank", "mbbank", "acb", "eximbank", "tpbank", "hdbank",
]
KEYWORD_PATTERN = re.compile("|".join(KEYWORDS), re.IGNORECASE)

MAX_ARTICLES = 20000  # điều chỉnh tuỳ thời gian/tài nguyên có sẵn
MIN_LEN = 200          # bỏ các đoạn quá ngắn (thường là rác/spam)

def is_finance_related(example):
    text = example["text"]
    return len(text) >= MIN_LEN and bool(KEYWORD_PATTERN.search(text))

news_stream = load_dataset("vietgpt/binhvq_news_vi", split="train", streaming=True)
filtered_stream = filter(is_finance_related, news_stream)

finance_articles = []
for i, example in enumerate(itertools.islice(filtered_stream, MAX_ARTICLES)):
    finance_articles.append(example["text"])
    if (i + 1) % 2000 == 0:
        print(f"Đã lọc được {i + 1} bài...")

print(f"\nTổng số bài viết banking/finance thu được: {len(finance_articles)}")
print("\nVí dụ:", finance_articles[0][:300])


README.md:   0%|          | 0.00/507 [00:00<?, ?B/s]

Đã lọc được 2000 bài...
Đã lọc được 4000 bài...
Đã lọc được 6000 bài...
Đã lọc được 8000 bài...
Đã lọc được 10000 bài...
Đã lọc được 12000 bài...
Đã lọc được 14000 bài...
Đã lọc được 16000 bài...
Đã lọc được 18000 bài...
Đã lọc được 20000 bài...

Tổng số bài viết banking/finance thu được: 20000

Ví dụ: Tốc độ giảm lương mạnh hơn giảm nhân sự nên thu nhập trong quý 2 của Eximbank chỉ còn 31,8 triệu đồng/người/3 tháng và 10,6 triệu đồng/người/tháng, giảm mạnh so với con số 37 triệu đồng/người/3 tháng và 12,4 triệu đồng/người/tháng.


In [23]:
# --- 9.2 Chunk văn bản (giữ nguyên kỹ thuật đã dùng ở project RAG trước) ---

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

documents = [
    Document(page_content=text, metadata={"source": "binhvq_news_vi", "doc_id": i})
    for i, text in enumerate(finance_articles)
]
chunks = splitter.split_documents(documents)

print(f"Số bài viết: {len(documents)}")
print(f"Số chunks sau khi split: {len(chunks)}")


Số bài viết: 20000
Số chunks sau khi split: 20002


In [24]:
# --- 9.3 Embed bằng multilingual-e5 + index vào Chroma ---
# Lưu ý: multilingual-e5 yêu cầu prefix "passage: " cho document và "query: " cho câu hỏi.

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# Thêm prefix "passage: " theo đúng chuẩn E5 trước khi embed
for c in chunks:
    c.page_content = "passage: " + c.page_content

PERSIST_DIR = "/kaggle/working/chroma_banking_news"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name="vn_banking_news",
)

print(f"Đã index {len(chunks)} chunks vào Chroma tại {PERSIST_DIR}")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Đã index 20002 chunks vào Chroma tại /kaggle/working/chroma_banking_news


In [25]:
# --- 9.4 Demo truy vấn thử ---

def search(query: str, k: int = 3):
    results = vectorstore.similarity_search("query: " + query, k=k)
    for r in results:
        print("-", r.page_content[:200].replace("passage: ", ""), "...")
        print()

search("lãi suất tiết kiệm ngân hàng hiện nay như thế nào")


- Theo biểu lãi suất mới của Ngân hàng TMCP Sài Gòn Thương Tín (Sacombank), lãi suất tiết kiệm có kỳ hạn thông thường giảm 0,1% xuống 4,7%/năm đối với kỳ hạn 1 tháng; lãi suất tiền gửi trực tuy ...

- Do lượng vốn hiện khá dồi dào, các ngân hàng thương mại đã đồng loạt giảm lãi suất huy động khoảng 0,2-0,3 điểm phần trăm, lãi suất tiết kiệm dao động từ 6,5-6,8%/năm các kỳ hạn dưới 6 tháng, ...

- Cũng theo người này, với tiền gửi tiết kiệm, lãi suất được đề nghị bằng lãi suất tiền vay được Chính phủ ban hành 4,8% một năm, như vậy, việc phải gửi tiền tiết kiệm cũng không ảnh hưởng tới  ...



In [26]:
# --- 9.5 Đóng gói để mang sang bước deploy (agent + FastAPI) ---
# Cách 1: nén thư mục Chroma, lưu làm Kaggle Dataset output (Save Version > giữ /kaggle/working)
import shutil
shutil.make_archive("/kaggle/working/chroma_banking_news", "zip", PERSIST_DIR)
print("Đã nén Chroma persist dir thành chroma_banking_news.zip — tải về hoặc dùng làm Kaggle Dataset output.")

# Cách 2 (khuyên dùng khi build agent thật): copy toàn bộ block '9.1' -> '9.3' thành 1 script
# riêng (build_rag_index.py), chạy 1 lần lúc deploy để tạo lại Chroma index từ đầu — tránh
# phải tải file zip lớn qua lại giữa các môi trường.


Đã nén Chroma persist dir thành chroma_banking_news.zip — tải về hoặc dùng làm Kaggle Dataset output.
